In [ ]:
import sys
import os

# Use absolute paths to ensure imports work from anywhere
pipeline_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline"
eda_dir = r"C:\Users\alexnewnham\Dropbox\MobRes\Scripts\CASD\Sandbox\Masslayoff\pipeline\eda"

sys.path.insert(0, eda_dir)
sys.path.insert(0, pipeline_dir)

import polars as pl
import numpy as np
import pandas as pd
from config import *
from eda_base import *
from eda_stats import *

# S5 Panel Data: Displaced Workers
Worker panel with pre/post mass layoff outcomes

In [ ]:
# Load displaced worker panel
displaced_path = os.path.join(OUT_DIR, 'displaced_panel.parquet')
if os.path.exists(displaced_path):
    displaced = pl.read_parquet(displaced_path)
    print(f"Displaced workers panel: {displaced.shape}")
    print(f"\nColumns (showing first 30): {displaced.columns[:30]}")
else:
    print(f"File not found: {displaced_path}")

In [ ]:
# Sample size and coverage
if 'displaced' in locals():
    n_workers = displaced.select(pl.col('ident_all').n_unique()).to_series()[0]
    n_obs = len(displaced)
    
    print(f"Total observations: {n_obs:,}")
    print(f"Unique workers: {n_workers:,}")
    print(f"Avg observations per worker: {n_obs / n_workers:.1f}")
    
    # Panel structure: years per worker
    if 'an' in displaced.columns:
        years_per_worker = displaced.group_by('ident_all').agg(
            pl.col('an').n_unique().alias('n_years')
        )
        
        print(f"\nYears per worker: min={years_per_worker['n_years'].min()}, max={years_per_worker['n_years'].max()}, mean={years_per_worker['n_years'].mean():.1f}")

In [ ]:
# Relative time structure: before/at/after layoff
if 'displaced' in locals() and 'yr_rel' in displaced.columns:
    yr_rel_dist = displaced.group_by('yr_rel').agg(
        pl.len().alias('n_obs')
    ).sort('yr_rel')
    
    print("Observations by year relative to layoff:")
    print(yr_rel_dist)
    
    # Pre/at/post split
    n_pre = displaced.select((pl.col('yr_rel') < 0).sum()).to_series()[0]
    n_at = displaced.select((pl.col('yr_rel') == 0).sum()).to_series()[0]
    n_post = displaced.select((pl.col('yr_rel') > 0).sum()).to_series()[0]
    
    print(f"\nPre-layoff (yr_rel < 0): {n_pre:,}")
    print(f"At layoff (yr_rel = 0): {n_at:,}")
    print(f"Post-layoff (yr_rel > 0): {n_post:,}")

In [ ]:
# Employment outcomes: wages before and after layoff
if 'displaced' in locals() and 's_brut' in displaced.columns and 'yr_rel' in displaced.columns:
    # Pre vs post wage comparison
    pre_layoff = displaced.filter(pl.col('yr_rel') == -1)
    post_layoff = displaced.filter(pl.col('yr_rel') > 0)
    
    pre_wage_mean = pre_layoff.select(pl.col('s_brut').mean()).to_series()[0]
    post_wage_mean = post_layoff.select(pl.col('s_brut').mean()).to_series()[0]
    
    print("Gross wage (s_brut) before and after layoff:")
    print(f"  Pre-layoff mean (yr_rel=-1): {pre_wage_mean:,.0f}")
    print(f"  Post-layoff mean (yr_rel>0): {post_wage_mean:,.0f}")
    
    if pre_wage_mean > 0:
        pct_change = (post_wage_mean - pre_wage_mean) / pre_wage_mean * 100
        print(f"  Change: {pct_change:.1f}%")

In [ ]:
# Hours worked
if 'displaced' in locals() and 'nbheur' in displaced.columns and 'yr_rel' in displaced.columns:
    pre_hours = displaced.filter(pl.col('yr_rel') == -1).select(pl.col('nbheur').mean()).to_series()[0]
    post_hours = displaced.filter(pl.col('yr_rel') > 0).select(pl.col('nbheur').mean()).to_series()[0]
    
    print("Hours worked (nbheur):")
    print(f"  Pre-layoff mean: {pre_hours:.0f}")
    print(f"  Post-layoff mean: {post_hours:.0f}")
    
    if pre_hours > 0:
        pct_change = (post_hours - pre_hours) / pre_hours * 100
        print(f"  Change: {pct_change:.1f}%")

In [ ]:
# Worker characteristics at displacement
if 'displaced' in locals() and 'yr_rel' in displaced.columns:
    at_layoff = displaced.filter(pl.col('yr_rel') == -1)  # year before layoff
    
    # Age and tenure
    if 'age' in at_layoff.columns:
        age_mean = at_layoff.select(pl.col('age').mean()).to_series()[0]
        print(f"Age at layoff: {age_mean:.1f}")
    
    if 'firm_entry' in at_layoff.columns:
        # Tenure = layoff year - firm entry year
        tenure_mean = at_layoff.select(
            (pl.col('an') - pl.col('firm_entry')).mean()
        ).to_series()[0]
        print(f"Tenure at layoff: {tenure_mean:.1f} years")
    
    # Occupation
    if 'cs2_f_at_ml' in at_layoff.columns:
        cs2_dist = at_layoff.group_by('cs2_f_at_ml').agg(
            pl.len().alias('n')
        ).sort('n', descending=True).head(10)
        print(f"\nTop occupations at layoff (CS2):")
        print(cs2_dist)

In [ ]:
# Geographic distribution
if 'displaced' in locals() and 'comt_at_ml' in displaced.columns:
    workers_by_commune = displaced.filter(
        pl.col('yr_rel') == -1
    ).group_by('comt_at_ml').agg(
        pl.col('ident_all').n_unique().alias('n_workers')
    ).sort('n_workers', descending=True).head(10)
    
    print("Top 10 communes by number of displaced workers:")
    print(workers_by_commune)

In [ ]:
# Heterogeneous impacts: outcomes by pre-layoff wage
if 'displaced' in locals() and 's_brut' in displaced.columns and 'yr_rel' in displaced.columns:
    # Classify by pre-layoff wage
    at_layoff = displaced.filter(pl.col('yr_rel') == -1)
    wage_25 = at_layoff.select(pl.col('s_brut').quantile(0.25)).to_series()[0]
    wage_75 = at_layoff.select(pl.col('s_brut').quantile(0.75)).to_series()[0]
    
    # Compare post-layoff wages by pre-layoff category
    post_layoff = displaced.filter(pl.col('yr_rel') > 0)
    
    low_wage_workers = post_layoff.filter(pl.col('s_brut') < wage_25).select(
        pl.col('s_brut').mean()
    ).to_series()[0]
    
    high_wage_workers = post_layoff.filter(pl.col('s_brut') > wage_75).select(
        pl.col('s_brut').mean()
    ).to_series()[0]
    
    print(f"Heterogeneous post-layoff outcomes:")
    print(f"  Low-wage category (< p25): {low_wage_workers:,.0f}")
    print(f"  High-wage category (> p75): {high_wage_workers:,.0f}")